[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/magrilu/cv-dojo/blob/main/notebooks/two-views/fundamental-matrix.ipynb)

# The fundamental matrix

We will explore epipolar geometry, the geometric and algebraic constraint
that ties two uncalibrated views of the same 3D scene.

We start from a question. Given a point $\mathbf{x}$ in the first image and a
point $\mathbf{x}'$ in the second, how do we decide whether they are the images of
the same 3D point $\mathbf{X}$?

The answer is a condition involving only the two image points and the two
cameras:

$$
\mathbf{x}'^\top \mathsf{F}\mathbf{x}=0.
$$

We first characterise corresponding points algebraically. We then revisit the
same condition geometrically and introduce the **fundamental matrix** as the map
that sends a point in one image to its epipolar line in the other.

As usual, the running examples are built around the Origami House. We start in a
noiseless setting here. Estimating $\mathsf F$ from noisy image correspondences is the
subject of the next notebook.

In [ ]:
#| echo: false
import sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules and not Path("cv-dojo").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/magrilu/cv-dojo.git"], check=True)
    import os
    os.chdir("cv-dojo")

for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src" / "cvdojo").exists():
        sys.path.insert(0, str(parent / "src"))
        break

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

from cvdojo.house import (load_image, load_model, load_two_view_cameras,
                          load_annotation)
from cvdojo.plotting import (clip_line_to_image, pairwise_intersections, ACCENT)
from cvdojo.scene import skew, set_axes_equal

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (8, 5), "axes.grid": False})

BLUE, GREY = "#3288BD", "0.45"

model = load_model()
V3 = {k: np.asarray(v, float).reshape(3, 1)
      for k, v in model["vertices"].items()}
EDGES = model["edges"]
IDS = list(V3)
index = {k: i for i, k in enumerate(IDS)}

# Points are columns throughout the notebook.
X = np.column_stack([V3[k] for k in IDS])          # 3 x N
P, Pp = load_two_view_cameras()


def homogeneous(x):
    """Add one homogeneous coordinate to points stored as columns."""
    return np.vstack((x, np.ones((1, x.shape[1]))))


def project_points(X, P):
    """Project 3D points stored as columns through a 3 x 4 camera matrix."""
    xh = P @ homogeneous(X)
    return xh[:2, :] / xh[2:3, :]


def camera_centre(P):
    """Euclidean camera centre as a 3 x 1 column vector."""
    _, _, Vt = np.linalg.svd(P)
    Ch = Vt[-1:, :].T
    Ch = Ch / Ch[-1, 0]
    return Ch[:3, :]


x = project_points(X, P)                           # 2 x N
xp = project_points(X, Pp)                         # 2 x N
xh = homogeneous(x)                                # 3 x N
xph = homogeneous(xp)                              # 3 x N
C, Cp = camera_centre(P), camera_centre(Pp)

I = load_image("IMG_4331.jpeg")
Ip = load_image("IMG_4337.jpeg")
H_IMG, W_IMG = I.shape[:2]

In [ ]:
#| echo: false
#| column: page
#| label: fig-two-photographs
#| fig-cap: >-
#|   The same origami house from two views. Are $\mathbf{x}_1$ and $\mathbf{x}_1'$ a pair of corresponding points?
_ann0 = load_annotation("two_view_matches")
_m = next(m for m in _ann0["matches"] if m["id"] == "X1")

fig, axes = plt.subplots(1, 2, figsize=(12, 7.5), layout="constrained")
for ax, im, p, lab, ttl in [
        (axes[0], I,  _m["x"],  r"$\mathbf{x}_1$",  "view 1"),
        (axes[1], Ip, _m["xp"], r"$\mathbf{x}'_1$", "view 2")]:
    ax.imshow(im)
    ax.scatter(*p, s=140, facecolors="none", edgecolors=ACCENT, lw=2.4, zorder=5)
    ax.text(p[0] + 34, p[1] - 34, lab, color=ACCENT, fontsize=15,
            bbox=dict(fc="white", alpha=0.75, ec="none", pad=2))
    ax.set_xlim(150, 1250); ax.set_ylim(1750, 700)
    ax.set_title(ttl, fontsize=10); ax.axis("off")
fig.suptitle("Are those corresponding points?", fontsize=11)
plt.show()


## When do two points correspond?

The next figure shows the Origami House and its exact projections through
two cameras, $\mathsf P$ and $\mathsf P'$. Ten vertices are marked so that the
same scene point can be followed across the two views.

In [ ]:
#| echo: false
#| column: page
#| label: fig-object-and-views
#| fig-cap: >-
#|   The ideal house in space, and its two exact perspective projections.
fig = plt.figure(figsize=(16, 5.4), layout="constrained")

# --- the object itself, in three dimensions -------------------------------
ax0 = fig.add_subplot(1, 3, 1, projection="3d")
for a, b in EDGES:
    ax0.plot(*zip(V3[a][:, 0], V3[b][:, 0]), color=GREY, lw=1.0)
ctr = X.mean(axis=1, keepdims=True)
for k, Q in V3.items():
    q = Q[:, 0]
    ax0.scatter(*q, s=18, color=ACCENT, depthshade=False, zorder=5)
    off = (Q - ctr)[:, 0]
    off[:2] = 0.55 * off[:2] / (np.linalg.norm(off[:2]) or 1)
    off[2] = 0.30 * np.sign(off[2])
    ax0.text(*(q + off), rf"$\mathbf{{X}}_{{{k[1:]}}}$",
             color=ACCENT, fontsize=9, ha="center", va="center")
d = model["dimensions_cm"]
ax0.set_box_aspect((d["long_side"], d["depth"], d["total_height"]))
ax0.view_init(elev=18, azim=-62)
ax0.set_axis_off()
ax0.set_title("the object\n$\\mathbf{X}_i \\in \\mathbb{P}^3$", fontsize=11)

# --- and its two images ---------------------------------------------------
for j, (pts, ttl, prime) in enumerate([
        (x,  r"view 1:  $\mathbf{x}_i = \mathsf{P}\,\mathbf{X}_i$", False),
        (xp, r"view 2:  $\mathbf{x}'_i = \mathsf{P}'\mathbf{X}_i$", True)]):
    ax = fig.add_subplot(1, 3, j + 2)
    Q = {k: pts[:, [index[k]]] for k in IDS}
    for a, b in EDGES:
        ax.plot([Q[a][0, 0], Q[b][0, 0]], [Q[a][1, 0], Q[b][1, 0]],
                color=GREY, lw=0.8)
    ax.scatter(pts[0, :], pts[1, :], s=40, facecolors="none",
               edgecolors=ACCENT, lw=1.6, zorder=5)
    for k, p in Q.items():
        lab = (rf"$\mathbf{{x}}'_{{{k[1:]}}}$" if prime
               else rf"$\mathbf{{x}}_{{{k[1:]}}}$")
        ax.text(p[0, 0] + 16, p[1, 0] - 16, lab, color=BLUE, fontsize=9)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title(ttl, fontsize=11)
    ax.set_xlabel("u  [px]")
    ax.set_ylabel("v  [px]")
plt.show()

Let us start from a definition of corresponding points.

::: {#def-correspondence}
## Corresponding points

Two image points $\mathbf{x}$ and $\mathbf{x}'$ are **corresponding points** if
there exists a world point $\mathbf{X}$ that projects to both:

$$
\lambda\mathbf{x}=\mathsf{P}\mathbf{X},
\qquad
\lambda'\mathbf{x}'=\mathsf{P}'\mathbf{X}.
$$
:::

The scale factors $\lambda$ and $\lambda'$ appear because projective points are
defined only up to scale.


In [ ]:
#| echo: false
#| column: margin
#| label: fig-two-rays
#| fig-cap: >-
#|   The viewing rays of a true correspondence meet at the same
#|   scene point $\mathbf X$.

Xk = V3[k]

fig = plt.figure(figsize=(3.3, 3.6))
ax = fig.add_subplot(111, projection="3d")
ax.set_proj_type("ortho")

# Origami House.
for a, b in EDGES:
    ax.plot(
        *zip(V3[a][:, 0], V3[b][:, 0]),
        color=GREY,
        lw=0.9,
    )

# Camera centres and viewing rays.
for Cc, lab in [(C, r"$\mathsf{C}$"),
                (Cp, r"$\mathsf{C}'$")]:
    ax.scatter(
        *Cc[:, 0],
        s=32,
        facecolor="white",
        edgecolor=BLUE,
        linewidth=1.5,
        depthshade=False,
    )
    ax.text(
        *(Cc[:, 0] + np.array([0.3, 0.3, 0.3])),
        lab,
        color=BLUE,
        fontsize=10,
    )
    ax.plot(
        *zip(Cc[:, 0], Xk[:, 0]),
        color=ACCENT,
        lw=1.6,
    )

# Common scene point.
ax.scatter(
    *Xk[:, 0],
    s=45,
    facecolor="white",
    edgecolor=ACCENT,
    linewidth=1.7,
    depthshade=False,
)
ax.text(
    *(Xk[:, 0] + np.array([0.2, 0.2, 0.35])),
    r"$\mathbf{X}$",
    color=ACCENT,
    fontsize=10,
)

# Equal scale along x, y, z: do not distort the house.
set_axes_equal(ax)

# Keep a cubic 3D box, but enlarge it inside the figure.
ax.set_box_aspect((1, 1, 1), zoom=1.25)

ax.view_init(elev=16, azim=-64)
ax.set_axis_off()

fig.subplots_adjust(
    left=0.0,
    right=1.0,
    bottom=0.0,
    top=1.0,
)

plt.show()

We can collect the two projection equations into one homogeneous linear system
for $(\mathbf{X},\lambda,\lambda')$:

$$
\underbrace{\begin{bmatrix}
\mathsf{P} & -\mathbf{x} & \mathbf{0} \\[2pt]
\mathsf{P}' & \mathbf{0} & -\mathbf{x}'
\end{bmatrix}}_{\textstyle \mathsf{L}}
\begin{bmatrix}
\mathbf{X}\\ \lambda\\ \lambda'
\end{bmatrix}
=\mathbf{0}.
$$

The matrix $\mathsf L$ is $6\times6$. A genuine world point exists exactly when
this system has a non-zero solution, hence when

$$
\det\mathsf L=0.
$$ {#eq-detL}

Let us substitute one known correspondence, $\mathbf{x}_9\leftrightarrow
\mathbf{x}'_9$.

In [ ]:
def correspondence_matrix(P, Pp, x, xp):
    """Linear system for two image points to come from the same 3D point."""
    z = np.zeros((3, 1))
    return np.block([
        [P,  -x,  z],
        [Pp,  z, -xp],
    ])

k = "X9"
i = index[k]
xi = xh[:, [i]]
xpi = xph[:, [i]]

L = correspondence_matrix(P, Pp, xi, xpi)
print(f"rank(L) = {np.linalg.matrix_rank(L)}")
print(f"det(L)  = {np.linalg.det(L):.3e}")

The rank of $\mathsf L$ is five and its determinant is zero. Its null space is
therefore one-dimensional, and the first four entries of a null vector give the
homogeneous coordinates of the world point. Let us recover it.

In [ ]:
_, _, Vt = np.linalg.svd(L)
V = Vt.T
# the solution is the last column of V
solution = V[:, [-1]]

# the reconstructed 3D point in homogeneous coordinate
Xh_rec = solution[:4, :]
# put in cartesian coordinates 
X_rec = Xh_rec[:3, :] / Xh_rec[3, 0]

print("recovered X:", np.round(X_rec[:, 0], 3), "cm")
print("model X:    ", np.round(V3[k][:, 0], 3), "cm")
print("difference: ", f"{np.linalg.norm(X_rec - V3[k]):.2e} cm")

As expected, the recovered point is $\mathbf{X}_9$. 


> What we have just done is **triangulation**: recovering a 3D point from its
> projections in two views. We will return to it in
> [Projective structure from two views](../reconstruction/triangulation.ipynb).

Now keep $\mathbf{x}_9$ fixed but pair it with the image of a different vertex
in the second view. The two points should no longer admit a common 3D preimage.

In [ ]:
xpi_wrong = xph[:, [index["X8"]]]

L_bad = correspondence_matrix(P, Pp, xi, xpi_wrong)
print(f"rank(L) = {np.linalg.matrix_rank(L_bad)}")
print(f"det(L)  = {np.linalg.det(L_bad):.3e}")

This time $\mathsf L$ has full rank. There is no non-zero solution, so the two
image points cannot be projections of the same world point through these
cameras.

We therefore have an algebraic correspondence test: corresponding points satisfy
@eq-detL. The determinant depends on the two cameras and on the two image points,
but the world point $\mathbf X$ has disappeared.

To be more precise, the point $\mathbf{x}$ appears in one
column of $\mathsf L$ and $\mathbf{x}'$ in another. Since a determinant is
linear in each column, $\det\mathsf L$ is linear in $\mathbf{x}$ and linear in
$\mathbf{x}'$: it is a **bilinear form**. Hence there is a $3\times3$ matrix
$\mathsf F$ such that

$$
\det\mathsf L = \mathbf{x}'^\top\mathsf F\mathbf{x}.
$$

Because the image points are homogeneous, $\mathsf F$ is defined only up to a
non-zero scale.

## The epipolar geometry

The same condition has a simple geometric interpretation. An image point
$\mathbf{x}$ determines a **visual ray** through the camera centre $\mathsf C$;
$\mathbf{x}'$ determines another ray through $\mathsf C'$.

The two image points correspond exactly when the two rays meet at a world point.

In [ ]:
#| echo: false
#| column: page
#| label: fig-epipolar-geometry
#| fig-cap: >-
#|   Epipolar geometry for one world point $\mathbf X$. The two camera centres
#|   and $\mathbf X$ define an epipolar plane. Its intersections with the image
#|   planes are the epipolar lines, which pass through the epipoles
#|   $\mathbf e$ and $\mathbf e'$ on the baseline.
RED = "#C0392B"


def unit(v):
    """Normalize a column vector."""
    return v / np.linalg.norm(v)


def image_plane(Cc, target, depth=3.0, half_width=4.0, half_height=2.7):
    """A small image plane in front of a camera centre."""
    z_axis = unit(target - Cc)
    up = np.array([[0.0], [0.0], [1.0]])
    x_axis = unit(np.cross(z_axis[:, 0], up[:, 0])[:, None])
    y_axis = np.cross(x_axis[:, 0], z_axis[:, 0])[:, None]

    centre = Cc + depth * z_axis
    corners = np.hstack((
        centre - half_width*x_axis - half_height*y_axis,
        centre + half_width*x_axis - half_height*y_axis,
        centre + half_width*x_axis + half_height*y_axis,
        centre - half_width*x_axis + half_height*y_axis,
    ))
    return centre, z_axis, corners


def ray_plane_intersection(Cc, Q, plane_point, plane_normal):
    """Intersect the ray Cc--Q with a plane."""
    direction = Q - Cc
    mu = ((plane_normal.T @ (plane_point - Cc)) /
          (plane_normal.T @ direction)).item()
    return Cc + mu * direction


# A controlled two-camera configuration, chosen only for the schematic.
Cd  = np.array([[-6.0], [-4.0], [1.0]])
Cpd = np.array([[ 6.0], [-4.0], [1.0]])
Xd  = np.array([[ 0.0], [ 2.0], [4.0]])

O1, n1, PL1 = image_plane(Cd,  Xd)
O2, n2, PL2 = image_plane(Cpd, Xd)

# Image points and epipoles are intersections with the two image planes.
Y1 = ray_plane_intersection(Cd,  Xd,  O1, n1)
Y2 = ray_plane_intersection(Cpd, Xd,  O2, n2)
E1 = ray_plane_intersection(Cd,  Cpd, O1, n1)
E2 = ray_plane_intersection(Cpd, Cd,  O2, n2)

fig = plt.figure(figsize=(12, 4.8))
ax = fig.add_subplot(111, projection="3d")

# A finite patch of the epipolar plane.
plane_patch = np.hstack((Cd, Cpd, Xd)).T
ax.add_collection3d(Poly3DCollection(
    [plane_patch], alpha=.12, facecolor=ACCENT, edgecolor="none"))

# The two image planes.
for plane in (PL1, PL2):
    ax.add_collection3d(Poly3DCollection(
        [plane.T], alpha=.11, facecolor=BLUE, edgecolor=BLUE, lw=1.1))

# Faint viewing pyramids only to make the two cameras legible.
for Cc, plane in ((Cd, PL1), (Cpd, PL2)):
    for j in range(plane.shape[1]):
        ax.plot(*zip(Cc[:, 0], plane[:, j]), color="0.78", lw=.7)

# Baseline and the two viewing rays.
ax.plot(*zip(Cd[:, 0], Cpd[:, 0]), color=BLUE, lw=2.2)
ax.plot(*zip(Cd[:, 0], Xd[:, 0]),  color="0.20", lw=1.4)
ax.plot(*zip(Cpd[:, 0], Xd[:, 0]), color="0.20", lw=1.4)

# Epipolar lines: the intersections of the epipolar plane with the image planes.
for E, Y in ((E1, Y1), (E2, Y2)):
    direction = Y - E
    A = E - .12 * direction
    B = Y + .22 * direction
    ax.plot(*zip(A[:, 0], B[:, 0]), color=RED, lw=3.0)

# Mathematical labels only: the definitions are given in the text above.
labels = (
    (Cd,  r"$\mathsf{C}$",        BLUE,   [-.15, 0.0, .65]),
    (Cpd, r"$\mathsf{C}'$",       BLUE,   [ .10, 0.0, .65]),
    (E1,  r"$\mathbf{e}$",        BLUE,   [-.45,-.10,-.50]),
    (E2,  r"$\mathbf{e}'$",       BLUE,   [ .25,-.10,-.50]),
    (Y1,  r"$\mathbf{x}$",        RED,    [-.30, .10, .55]),
    (Y2,  r"$\mathbf{x}'$",       RED,    [ .20, .10, .55]),
    (Xd,  r"$\mathbf{X}$",        "0.15", [ .00, .10, .55]),
)
for Q, label, colour, offset in labels:
    q = Q[:, 0]
    ax.scatter(*q, s=48, facecolors="white", edgecolors=colour,
               linewidths=1.8, depthshade=False, zorder=8)
    ax.text(*(q + np.asarray(offset)), label, color=colour,
            fontsize=14, zorder=9)

# Zoom on the geometry that matters; no house or distant epipoles set the limits.
pts = np.hstack((PL1, PL2, Cd, Cpd, Xd, E1, E2, Y1, Y2)).T
lo, hi = pts.min(axis=0), pts.max(axis=0)
span = hi - lo
margin = np.array([.04, .08, .12]) * span
lo, hi = lo - margin, hi + margin

ax.set_xlim(lo[0], hi[0])
ax.set_ylim(lo[1], hi[1])
ax.set_zlim(lo[2], hi[2])
ax.set_box_aspect(hi - lo, zoom=1.35)
ax.view_init(elev=18, azim=-73)
ax.set_axis_off()
fig.subplots_adjust(0, 0, 1, 1)
plt.show()


Three points are now in play: the two camera centres and the world point. Some names:

::: {#def-epipolar-plane}
## Epipolar plane and baseline

The plane through $\mathsf{C}$, $\mathsf{C}'$ and $\mathbf{X}$ is the
**epipolar plane** of $\mathbf{X}$.

The line $\mathsf{C}\mathsf{C}'$ is the **baseline**. It is the same for every
world point, so every epipolar plane contains it: they form a pencil of planes
hinged on the baseline.
:::

::: {#def-epipoles}
## Epipoles

The baseline meets the first image plane in a point $\mathbf{e}$ and the second
in $\mathbf{e}'$: the **epipoles**. Equivalently, $\mathbf{e}$ is the image of
the second camera centre in the first view, and $\mathbf{e}'$ the image of the
first centre in the second view.
:::

::: {#def-epipolar-line}
## Epipolar lines

The epipolar plane cuts each image plane in a line: the **epipolar lines**
$\boldsymbol{\ell}$ and $\boldsymbol{\ell}'$. Every epipolar line passes through
the epipole of its image, since the baseline lies in every epipolar plane.
:::


Fix $\mathbf{x}$ in the first image. Its visual ray lies in one epipolar plane,
and so does every possible world point that projects to $\mathbf{x}$. Therefore
the corresponding point $\mathbf{x}'$ is **not free** to vary over the second
image: it must lie on the epipolar line cut out by that plane.

We can construct this line without knowing $\mathbf{X}$. Two points on the first
camera's back-projected line are enough: the camera centre $\mathsf C$ and, for
example, $\mathsf P^+\mathbf{x}$. Projecting both into the second view and
joining them gives

$$
\begin{aligned}
\boldsymbol\ell'
&=(\mathsf P'\mathsf C)\times(\mathsf P'\mathsf P^+\mathbf{x})\\
&=\mathbf e'\times(\mathsf P'\mathsf P^+\mathbf{x})\\
&=\underbrace{[\mathbf e']_\times\mathsf P'\mathsf P^+}_{\textstyle\mathsf F}
\mathbf{x}.
\end{aligned}
$$

Thus $\mathsf F$ maps a point in the first image to its **epipolar line** in the
second. On our exact correspondences, every $\mathbf{x}'_i$ should lie on
$\mathsf F\mathbf{x}_i$.

In [ ]:
# Pseudoinverse of the first camera.
P_pinv = np.linalg.pinv(P)

# Project the first camera centre into the second image.
C_h = np.vstack((C, [[1.0]]))
e_prime = Pp @ C_h

# Fundamental matrix: F = [e']_x P' P^+.
F = skew(e_prime[:, 0]) @ Pp @ P_pinv
F /= np.linalg.norm(F)

The epipolar constraint can also be checked geometrically. For each point
$\mathbf x_i$ in the first image, the fundamental matrix gives the
corresponding epipolar line in the second image,

$$
\boldsymbol\ell'_i = \mathsf F \mathbf x_i.
$$

If $\mathbf x'_i$ is the corresponding point, it should lie on this line.
For a line $\boldsymbol\ell'=(a,b,c)^\top$, the Euclidean distance of
$\mathbf x'=(u',v',1)^\top$ from the line is

$$
d(\mathbf x',\boldsymbol\ell')
=
\frac{|\mathbf x'^\top \boldsymbol\ell'|}
{\sqrt{a^2+b^2}}.
$$

Let us evaluate this distance for all the correspondences of the Origami House.

In [ ]:
# Epipolar line l' = F x for each point in the first image.
lines_prime = F @ xh

# Normalize the Euclidean normal (a, b) of each line.
line_norms = np.linalg.norm(lines_prime[:2, :], axis=0)

# Distance of each x' from its corresponding epipolar line.
distances = (
    np.abs(np.sum(xph * lines_prime, axis=0))
    / line_norms
)

for name, distance in zip(IDS, distances):
    print(f"{name}: {distance:.2e} px")

The distances vanish to machine precision, as expected for exact data. The point-to-epipolar-line distance is only one way of measuring how well a
correspondence satisfies the epipolar constraint. Other
 residuals are treated in
[Epipolar residuals](../two-views/residuals.ipynb).

Now consider the map

$$
\begin{array}{ccc}
\mathbb P^2 & \dashrightarrow & (\mathbb P^2)^*\\[2mm]
\mathbf x & \longmapsto & \mathsf F\mathbf x .
\end{array}
$$

It sends points of $\mathbb P^2$ to lines of the dual plane $\mathbb (P^2)^*$. The map is not
one-to-one: any two points on the same line through the epipole belong to the
same epipolar plane, and therefore generate the same epipolar line in the other
view.

In [ ]:
# The first epipole is the right null vector of F.
U, S, Vt = np.linalg.svd(F)
e = Vt.T[:, [-1]]
e /= e[2, 0]

x_a = xh[:, [index["X9"]]]
# another point on the very same line through e and x_a
x_b = e + 0.35 * (x_a - e)     

l_a = F @ x_a
l_b = F @ x_b

l_a /= np.linalg.norm(l_a[:2])
l_b /= np.linalg.norm(l_b[:2])

# Homogeneous lines may differ by a global sign.
gap = min(
    np.linalg.norm(l_a - l_b),
    np.linalg.norm(l_a + l_b),
)

print("x_a =", np.round(x_a[:2, 0], 1))
print("x_b =", np.round(x_b[:2, 0], 1))
print("difference between their epipolar lines:", f"{gap:.2e}")

Two different points in input, the very same line as output. The map is
therefore not one-to-one and, since $\mathsf F$ is linear,

$$
\operatorname{rank}\mathsf F = 2,
\qquad
\det\mathsf F = 0.
$$

We could have seen the rank deficiency already from the construction

$$
\mathsf F = [\mathbf e']_\times \mathsf P'\mathsf P^+,
$$

because the skew-symmetric matrix $[\mathbf e']_\times$ has rank two. Hence
$\mathsf F$ has rank at most two, and for a non-degenerate pair of cameras its
rank is exactly two.

The missing dimension has a clear geometric meaning. In the first image,

$$
\mathsf F\mathbf e=\mathbf 0.
$$

The epipole $\mathbf e$ is the one point for which no epipolar line is defined:
its viewing ray is the baseline itself, so there is no unique epipolar plane.
Similarly, in the second image,

$$
\mathbf e'^\top\mathsf F=\mathbf 0.
$$

Thus $\mathbf e$ spans the right nullspace of $\mathsf F$, while $\mathbf e'$
spans its left nullspace. Since both nullspaces are one-dimensional, a single
SVD

$$
\mathsf F = U\Sigma V^\top
$$

gives both epipoles: $\mathbf e$ is the last column of $V$, and $\mathbf e'$ is
the last column of $U$.


Since $\mathsf F$ has rank two, it has a one-dimensional right nullspace and a
one-dimensional left nullspace. These are precisely the two epipoles,

$$
\mathsf F\mathbf e = \mathbf 0,
\qquad
\mathbf e'^\top\mathsf F = \mathbf 0.
$$

A single SVD therefore gives us both.

In [ ]:
# The epipoles are the right and left null vectors of F.
U, S, Vt = np.linalg.svd(F)

e       = Vt.T[:, [-1]]
e_prime = U[:, [-1]]

# Choose representatives with last homogeneous coordinate equal to one.
e       /= e[2, 0]
e_prime /= e_prime[2, 0]

print("e  =", np.round(e[:, 0], 2))
print("e' =", np.round(e_prime[:, 0], 2))

In [ ]:
# The first epipole is also the image of the second camera centre.
Cp_h = np.vstack((Cp, [[1.0]]))

e_from_camera = P @ Cp_h
e_from_camera /= e_from_camera[2, 0]

print("P C' =", np.round(e_from_camera[:, 0], 2))

## Special configurations

The previous construction holds for any pair of cameras, but particular relative
motions leave a visible signature in $\mathsf F$.

Using the Origami House as a synthetic scene, we can place two virtual cameras in
simple configurations and inspect the resulting epipolar geometry.

In [ ]:
#| echo: false
def virtual_pair(direction, baseline=6.0, distance=34.0, f=1800.0,
                 W=1400, H=1050):
    """Parallel synthetic cameras separated along a camera-frame direction."""
    ctr = X.mean(axis=1, keepdims=True)
    v = np.array([[0.9], [-1.0], [0.55]])
    look = ctr + distance * v / np.linalg.norm(v)

    z_axis = ctr - look
    z_axis /= np.linalg.norm(z_axis)
    x_axis = np.cross(np.array([[0.0], [0.0], [-1.0]])[:, 0],
                      z_axis[:, 0])[:, None]
    x_axis /= np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis[:, 0], x_axis[:, 0])[:, None]
    R = np.vstack((x_axis.T, y_axis.T, z_axis.T))

    K = np.array([[f, 0.0, W / 2],
                  [0.0, f, H / 2],
                  [0.0, 0.0, 1.0]])

    C1 = look
    direction = np.asarray(direction, float).reshape(3, 1)
    C2 = look + baseline * (R.T @ direction)

    def make_camera(Cc):
        return K @ np.hstack((R, -R @ Cc))

    return make_camera(C1), make_camera(C2), W, H


def fundamental_from_cameras(P, Pp):
    C = camera_centre(P)
    ep = Pp @ homogeneous(C)
    F = skew(ep[:, 0]) @ Pp @ np.linalg.pinv(P)
    return F / np.linalg.norm(F)

In [ ]:
# Two camera pairs with the same orientation but different translation directions:
# sideways motion along the x-axis, and forward motion along the optical axis.
P_side, Pp_side, W_v, H_v = virtual_pair([1, 0, 0])
P_fwd,  Pp_fwd,  _,   _   = virtual_pair([0, 0, 1])

# Fundamental matrices induced by the two camera configurations.
F_side = fundamental_from_cameras(P_side, Pp_side)
F_fwd  = fundamental_from_cameras(P_fwd, Pp_fwd)

# F is defined only up to scale, so normalize it for display.
F_side /= np.abs(F_side).max()
F_fwd  /= np.abs(F_fwd).max()

print("Sideways motion:")
print(np.round(F_side, 3))

print("\nForward motion:")
print(np.round(F_fwd, 3))

In [ ]:
#| echo: false
#| column: page
#| label: fig-special-configurations
#| fig-cap: >-
#|   Two motions, in synthetic views of the same house. **Left:** the cameras
#|   move sideways and remain parallel, so the epipolar lines are horizontal
#|   and the epipoles lie at infinity. **Right:** the cameras move straight
#|   ahead, so all epipolar lines pass through the same finite epipole, the
#|   focus of expansion.

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5.2),
    layout="constrained"
)

configurations = [
    (P_side, Pp_side, F_side, "sideways motion: a rectified pair"),
    (P_fwd,  Pp_fwd,  F_fwd,  "forward motion: the epipole stays put"),
]

for ax, (Pa, Pb, Fm, title) in zip(axes, configurations):

    # Project the Origami House into the two virtual views.
    a = project_points(X, Pa)
    b = project_points(X, Pb)

    # Draw the house in the second view.
    Q = {name: b[:, [index[name]]] for name in IDS}
    for u, v in EDGES:
        ax.plot(
            [Q[u][0, 0], Q[v][0, 0]],
            [Q[u][1, 0], Q[v][1, 0]],
            color=GREY,
            lw=1.4,
            zorder=3,
        )

    # Epipolar lines l' = F x in the second view.
    lines = Fm @ homogeneous(a)

    for j in range(lines.shape[1]):
        segment = clip_line_to_image(lines[:, j], W_v, H_v)

        if segment is not None:
            ax.plot(
                segment[:, 0],
                segment[:, 1],
                color=ACCENT,
                lw=1.0,
                alpha=.9,
                zorder=2,
            )

    # Corresponding image points.
    ax.scatter(
        b[0, :],
        b[1, :],
        s=34,
        facecolors="none",
        edgecolors=BLUE,
        lw=1.6,
        zorder=5,
    )

    # The second epipole e' is the left null vector of F.
    U, _, _ = np.linalg.svd(Fm)
    e_prime = U[:, [-1]]

    # A finite epipole can be normalized to (u, v, 1).
    finite = abs(e_prime[2, 0]) > 1e-12

    if finite:
        e_prime /= e_prime[2, 0]

        if (
            0 <= e_prime[0, 0] < W_v
            and 0 <= e_prime[1, 0] < H_v
        ):
            ax.scatter(
                e_prime[0, 0],
                e_prime[1, 0],
                s=90,
                color=BLUE,
                edgecolors="white",
                lw=1.4,
                zorder=6,
            )
            ax.text(
                e_prime[0, 0] + 26,
                e_prime[1, 0] + 46,
                r"$\mathbf{e}=\mathbf{e}'$",
                color=BLUE,
                fontsize=13,
            )

    ax.set_xlim(0, W_v)
    ax.set_ylim(H_v, 0)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=12)
    ax.axis("off")

plt.show()

**Sideways motion.** With the same orientation and a translation along the camera
$x$ axis, the epipolar constraint reduces to $v=v'$. The epipolar lines are
horizontal, the epipoles move to infinity, and a match can only move along its
own row. This is a **rectified pair**: a two-dimensional correspondence search
has become one-dimensional.

**Forward motion.** With a translation along the optical axis, $\mathsf F$ is
skew-symmetric. The two epipoles coincide, $\mathbf e=\mathbf e'$, at the
**focus of expansion**.

## The pencil of epipolar planes

Every epipolar plane contains the baseline $\mathsf C\mathsf C'$. The epipolar
planes therefore form a **pencil of planes**: a one-parameter family of planes
rotating about the same line.

Each epipolar plane intersects the two image planes in a pair of corresponding
epipolar lines,

$$
\boldsymbol\ell
\qquad\longleftrightarrow\qquad
\boldsymbol\ell'.
$$

This gives another way to think about epipolar geometry: not only as a
point-to-line map, but also as a correspondence between two pencils of lines,
one through $\mathbf e$ and one through $\mathbf e'$.

Now choose any plane $\pi$ in the scene that does not pass through either
camera centre. It induces a homography $\mathsf H$ between the two images.

To see how this is related to epipolar geometry, start from an image point
$\mathbf x$ in the first view. Its viewing ray meets $\pi$ at some point
$\mathbf X_\pi$. By definition of the plane homography,

$$
\mathbf x_\pi'
\sim
\mathsf H\mathbf x
$$

is the image of $\mathbf X_\pi$ in the second view.

But $\mathbf X_\pi$ lies on the viewing ray of $\mathbf x$. Hence
$\mathsf C$, $\mathsf C'$ and $\mathbf X_\pi$ lie in the same epipolar plane
determined by $\mathbf x$. Its image $\mathbf x_\pi'$ must therefore lie on
the corresponding epipolar line $\boldsymbol\ell'$.

That line also passes through the epipole $\mathbf e'$. Thus it is simply the
line joining $\mathbf e'$ and $\mathsf H\mathbf x$:

$$
\boldsymbol\ell'
=
\mathbf e' \times (\mathsf H\mathbf x)
=
[\mathbf e']_\times \mathsf H\mathbf x.
$$

On the other hand, by definition of the fundamental matrix,

$$
\boldsymbol\ell'=\mathsf F\mathbf x.
$$

Since this holds for every $\mathbf x$,

$$
\mathsf F
\sim
[\mathbf e']_\times \mathsf H
$$ {#eq-fromH}

The same relation can be seen directly at the level of lines. Take one
epipolar plane and intersect it with the scene plane $\pi$. Their intersection
is a 3D line. Its two images are precisely the corresponding epipolar lines
$\boldsymbol\ell$ and $\boldsymbol\ell'$.

A homography $\mathbf x' \sim \mathsf H\mathbf x$ transforms lines according to

$$
\boldsymbol\ell'
\sim
\mathsf H^{-\top}\boldsymbol\ell,
$$ {#eq-linehomography}

so the plane homography carries the pencil of epipolar lines through
$\mathbf e$ onto the pencil through $\mathbf e'$.

We can see both relations on the Origami House by using the wall containing
the red door as the plane $\pi$.

In [ ]:
#| echo: false

def homography_from_four(a, b):
    """Plane homography from four column correspondences (DLT)."""
    A = []

    for i in range(a.shape[1]):
        u, v = a[:2, i]
        up, vp = b[:2, i]

        A += [
            [-u, -v, -1,  0,  0,  0, up*u, up*v, up],
            [ 0,  0,  0, -u, -v, -1, vp*u, vp*v, vp],
        ]

    _, _, Vt = np.linalg.svd(np.asarray(A))
    H = Vt[-1, :].reshape(3, 3)

    return H / H[2, 2]


DOOR_WALL = ["X0", "X1", "X5", "X4"]
idx = [index[k] for k in DOOR_WALL]

H_door = homography_from_four(
    xh[:, idx],
    xph[:, idx],
)

# Check that H maps the four wall points from view 1 to view 2.
q = H_door @ xh[:, idx]
q = q[:2, :] / q[2:3, :]

assert np.max(np.abs(q - xp[:, idx])) < 1e-8


# Compute e' locally, directly from the camera geometry.
C_h = np.vstack((C, [[1.0]]))
e_prime_plane = Pp @ C_h

# Check F ~ [e']_x H.
F_from_plane = skew(e_prime_plane[:, 0]) @ H_door

F_from_plane /= np.linalg.norm(F_from_plane)
F_reference = F / np.linalg.norm(F)

# Resolve the arbitrary global sign.
if np.sum(F_from_plane * F_reference) < 0:
    F_from_plane = -F_from_plane

assert np.max(np.abs(F_from_plane - F_reference)) < 1e-8

In [ ]:
#| echo: false
#| column: page
#| label: fig-plane-homography
#| fig-cap: >-
#|   **Left:** four epipolar lines in the second view, $\boldsymbol\ell' =
#|   \mathsf F\mathbf{x}$, one colour each. **Right:** the same four lines
#|   carried back to the first view, drawn twice. In colour, through the
#|   fundamental matrix, $\boldsymbol\ell = \mathsf F^\top\mathbf{x}'$; in white
#|   dashes, through the wall with the door,
#|   $\boldsymbol\ell = \mathsf H^\top\boldsymbol\ell'$. The two agree.
import matplotlib.patheffects as pe
HALO = lambda w, c="0.15": [pe.Stroke(linewidth=w, foreground=c), pe.Normal()]

angles = np.array([
    np.arctan2((F @ xh[:, [i]])[1, 0], (F @ xh[:, [i]])[0, 0]) % np.pi
    for i in range(xh.shape[1])
])
pick = list(np.argsort(angles)[np.linspace(0, len(angles)-1, 4).astype(int)])
cols = plt.cm.Spectral(np.linspace(.08, .92, len(pick)))


def long_segment(ax, line, W, H, **kw):
    seg = clip_line_to_image(line[:, 0], W, H)
    if seg is None:
        return
    d = seg[1] - seg[0]
    d = d / np.linalg.norm(d)
    far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
    ax.plot(far[:, 0], far[:, 1], **kw)


fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6), layout="constrained")
axes[0].imshow(Ip)
for i, col in zip(pick, cols):
    long_segment(axes[0], F @ xh[:, [i]], W_IMG, H_IMG,
                 color=col, lw=2.0, zorder=3, path_effects=HALO(3.8))
    axes[0].scatter(xp[0, i], xp[1, i], s=55, facecolors="none",
                    edgecolors=col, lw=2.2, zorder=6,
                    path_effects=HALO(3.6, "white"))
axes[0].set_title(r"view 2:  $\boldsymbol{\ell}' = \mathsf{F}\mathbf{x}$",
                  fontsize=12)

axes[1].imshow(I)
for i, col in zip(pick, cols):
    line_p = F @ xh[:, [i]]
    long_segment(axes[1], F.T @ xph[:, [i]], W_IMG, H_IMG,
                 color=col, lw=2.0, zorder=3, path_effects=HALO(3.8))
    long_segment(axes[1], H_door.T @ line_p, W_IMG, H_IMG,
                 color="white", lw=1.7, ls=(0, (5, 4)), zorder=4,
                 path_effects=HALO(3.5))
    axes[1].scatter(x[0, i], x[1, i], s=55, facecolors="none",
                    edgecolors=col, lw=2.2, zorder=6,
                    path_effects=HALO(3.6, "white"))

poly = np.column_stack((x[:, idx], x[:, [idx[0]]]))
axes[1].plot(poly[0, :], poly[1, :], color=BLUE, lw=2.4, zorder=5,
             path_effects=HALO(4.2, "white"))
axes[1].set_title(r"view 1:  $\mathsf{F}^\top\mathbf{x}'$  against  "
                  r"$\mathsf{H}^\top\boldsymbol{\ell}'$", fontsize=12)

for ax, pts in zip(axes, (xp, x)):
    pad = 300
    ax.set_xlim(pts[0, :].min() - pad, pts[0, :].max() + pad)
    ax.set_ylim(pts[1, :].max() + pad, pts[1, :].min() - pad)
    ax.axis("off")
plt.show()

The two constructions give the same epipolar lines. The chosen plane was only
scaffolding: another scene plane would induce a different $\mathsf H$, but
$[\mathbf e']_\times\mathsf H$ would describe the same epipolar geometry.

There is an important exception. If the **entire scene is planar**, every
correspondence already satisfies $\mathbf{x}'\sim\mathsf H\mathbf{x}$. In that
case the fundamental matrix is not determined by the correspondences alone: a
first example of a critical configuration.

## $\mathsf F$ and $\mathsf F^\top$

The construction has treated the two images asymmetrically: we fixed $\mathbf x$
in the first and asked where $\mathbf x'$ could be. Nothing forces that order.
Running the same argument the other way round gives the epipolar line of
$\mathbf x'$ in the first image, and the matrix that does it is the transpose:

$$\boldsymbol\ell' = \mathsf F\,\mathbf x, \qquad
\boldsymbol\ell = \mathsf F^\top\mathbf x' .$$

So a single matrix carries the geometry in both directions, and swapping the
roles of the images means transposing it. Everything comes in pairs accordingly:
$\mathsf F\mathbf e = \mathbf 0$ and $\mathsf F^\top\mathbf e' = \mathbf 0$, the
epipole of one view being the null vector of one matrix and the epipole of the
other the null vector of its transpose.

In [ ]:
#| echo: false
#| column: body
#| label: fig-transpose
#| fig-cap: >-
#|   Transposing $\mathsf F$ swaps the roles of the two images: it carries a point
#|   of the second view to its epipolar line in the first.
lines = F.T @ xph
lines_rows = lines.T                              # plotting helper expects rows
pts = pairwise_intersections(lines_rows)
ctr = np.median(pts, axis=0)

fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
ax.imshow(I)
for j in range(lines.shape[1]):
    line = lines[:, j]
    seg = clip_line_to_image(line, W_IMG, H_IMG)
    if seg is None:
        continue
    d = seg[1] - seg[0]
    d = d / np.linalg.norm(d)
    far = np.vstack([seg[0] - 2600*d, seg[1] + 2600*d])
    ax.plot(far[:, 0], far[:, 1], color=ACCENT, lw=0.9)
ax.scatter(x[0, :], x[1, :], s=45, facecolors="none",
           edgecolors=BLUE, lw=1.6, zorder=5)
ax.scatter(*ctr, s=60, color=BLUE, zorder=6)
ax.text(ctr[0] + 40, ctr[1] - 40, r"$\mathbf{e}$", color=BLUE, fontsize=13)
ax.set_title(r"$\mathsf{F}^\top$ carries the second view's points"
             "\n" r"to epipolar lines in the first", fontsize=11)
ax.set_xlim(min(0, ctr[0]) - 250, max(W_IMG, ctr[0]) + 250)
ax.set_ylim(max(H_IMG, ctr[1]) + 250, min(700, ctr[1]) - 250)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## Fundamental matrices and cameras

We have reached $\mathsf F$ in two ways. Geometrically,

$$
\mathsf F=[\mathbf e']_\times\mathsf P'\mathsf P^+,
$$

by following a visual ray from one camera to the other. Algebraically,
$\mathbf{x}'^\top\mathsf F\mathbf{x}$ appeared by expanding the determinant in
@eq-detL.

The algebraic route gives another expression. Expanding by cofactors, each entry
of $\mathsf F$ is a $4\times4$ minor built from two rows of $\mathsf P$ and two
rows of $\mathsf P'$:

$$
\mathsf F_{ji}=(-1)^{i+j}
\det\!\begin{bmatrix}
\widetilde{\mathsf P}_i\\
\widetilde{\mathsf P}'_j
\end{bmatrix},
$$

where $\widetilde{\mathsf P}_i$ denotes $\mathsf P$ with row $i$ removed.
Thus the fundamental matrix can be constructed directly from the two cameras,
with no image correspondences. Its rank-two structure follows automatically.

In [ ]:
def F_from_minors(P, Pp):
    """Build F from the 4 x 4 minors of two camera matrices."""
    Fm = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            rows = ([P[k] for k in range(3) if k != i] +
                    [Pp[k] for k in range(3) if k != j])
            Fm[j, i] = (-1)**(i + j) * np.linalg.det(np.vstack(rows))
    return Fm / np.linalg.norm(Fm)


F_min = F_from_minors(P, Pp)
F_min *= np.sign(np.sum(F_min * F))

print("rank(F) =", np.linalg.matrix_rank(F_min))
print("agreement with geometric construction:",
      f"{np.max(np.abs(F_min - F)):.2e}")

# The determinant of L and x'^T F x differ only by one global scale.
for i, j in [(0, 5), (2, 7), (3, 9), (6, 1)]:
    Lij = correspondence_matrix(P, Pp, xh[:, [i]], xph[:, [j]])
    det_L = np.linalg.det(Lij)
    bilinear = (xph[:, [j]].T @ F_min @ xh[:, [i]]).item()
    print(f"{IDS[i]}/{IDS[j]}: det(L) / (x'^T F x) = {det_L / bilinear:.4e}")

## The projective reconstruction theorem

The fundamental matrix contains enough information to recover the two-view
geometry, but not a unique Euclidean scene.

::: {.callout-note title="Projective reconstruction theorem"}
Under the usual non-degeneracy assumptions, if a set of point correspondences
in two views determines the fundamental matrix uniquely, then the cameras and
the scene can be reconstructed from those correspondences alone.

Any two such reconstructions are related by a projective transformation of
$\mathbb P^3$.
:::

In other words, image correspondences determine the scene only **up to a
3D projectivity**. If one reconstruction is described by cameras
$\mathsf P,\mathsf P'$ and points $\mathbf X_i$, then for any invertible
$4\times4$ matrix $\mathsf H$,

$$
\widetilde{\mathsf P}  = \mathsf P\mathsf H^{-1},
\qquad
\widetilde{\mathsf P}' = \mathsf P'\mathsf H^{-1},
\qquad
\widetilde{\mathbf X}_i = \mathsf H\mathbf X_i
$$

produces exactly the same image measurements, since

$$
\widetilde{\mathsf P}\,\widetilde{\mathbf X}_i
= \mathsf P\mathbf X_i,
\qquad
\widetilde{\mathsf P}'\,\widetilde{\mathbf X}_i
= \mathsf P'\mathbf X_i.
$$

The theorem tells us what can be recovered from two uncalibrated views.
The next question is constructive: given $\mathsf F$, how can we choose one
pair of cameras and obtain one representative of this projective
reconstruction?

## Questions to leave open

**We built $\mathsf F$ from two known cameras. What if we only have the images?**
The constraint is linear in the entries of $\mathsf F$, so eight
correspondences give eight equations. Is the resulting matrix automatically a
fundamental matrix? This is the starting point of the next notebook.

**A general $\mathsf F$ has seven degrees of freedom, while special motions can
reduce them.** If we knew in advance that the motion were forward, could we
estimate $\mathsf F$ from fewer correspondences?

**$\mathsf F$ maps a point to a line.** Two views therefore constrain a match to
an epipolar line but do not say where on that line it lies. What extra
information would identify the corresponding point?

**The epipoles of our real pair lie outside the image, while forward motion puts
them near the centre.** How does the camera motion determine where the epipoles
appear?

## Further reading

- Longuet-Higgins, H. C. "A computer algorithm for reconstructing a scene from two projections", *Nature* 293, 1981. The original eight-point algorithm.
- Hartley, R. "In defense of the eight-point algorithm", *IEEE TPAMI* 19(6), 1997. Why normalizing the coordinates matters as much as it does.
- Hartley, R. and Zisserman, A. *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge University Press, 2004. Chapter 9 for the epipolar geometry, Chapter 11 for the computation of $\mathsf{F}$, and Result 17.1 for the minors.
- Fusiello, A. Computer Vision: *Three-dimensional Reconstruction Techniques*, Springer Cham, 2024.

---

**Luca Magri** — Computer Vision Dojo
Code MIT · text and figures CC BY-NC-ND 4.0
<https://magrilu.github.io/cv-dojo/>
